In [9]:
#Importación de sesión de Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

In [10]:
#Creación de sesión
spark = SparkSession \
    .builder \
    .appName("HousingAnalysis") \
    .getOrCreate()

In [11]:
#Carga de datos
path_archivo = '/Users/Nefi/OneDrive/Documentos/09-EBAC/EBACMX-DATA-ANALYST/Referencias/kc_house_data_clean.csv'
df = spark.read.csv(path_archivo, header=True, inferSchema=True)

In [12]:
#Validación de carga correcta de datos
print("Primeras filas del dataset:")
df.show(5)

print("Esquema de datos:")
df.printSchema()

Primeras filas del dataset:
+----------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|        id| price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|      date|sale_year|price_per_m2|
+----------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|7129300520|221900|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|2014-10-13|     2014|     2024.16|
|6414100192|538000|       3|     2.25|       2570|  

In [13]:
#Listado ordenado por zipcode
df_ordenado = df.orderBy(col("zipcode"))

In [14]:
#Visualización de datos ordenados por zipcode
print("Datos ordenados por zipcode:")
df_ordenado.show()

Datos ordenados por zipcode:
+----------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|        id| price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|      date|sale_year|price_per_m2|
+----------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+----------+---------+------------+
|3353400860|249900|       3|     1.75|       2080|   12522|   1.0|         0|   0|        5|    6|      2080|            0|    1950|           0|  98001| 47.267| -122.25|         1690|     11200|2014-07-17|     2014|     1293.22|
|3354400060|238000|       2|      1.0|       1088| 

In [15]:
#Zipocode con mayor cantidad de casas
zipcode_mayor = df.groupBy("zipcode") \
    .agg(count("*").alias("num_casas")) \
    .orderBy(col("num_casas").desc()) \
    .first()["zipcode"]

print(f"Zipcode con más casas: {zipcode_mayor}")

Zipcode con más casas: 98103


In [16]:
#Filtro para el zipcode obtenido
df_zip = df.filter(col("zipcode") == zipcode_mayor)

In [17]:
#Promedios de precio y tamaño
promedios = df_zip.agg(
    avg("price").alias("precio_promedio"),
    avg("sqft_living").alias("tamano_promedio_m2")
)

print("Promedios para el zipcode con más casas:")
promedios.show()

Promedios para el zipcode con más casas:
+-----------------+------------------+
|  precio_promedio|tamano_promedio_m2|
+-----------------+------------------+
|584919.2109634551|1650.8305647840532|
+-----------------+------------------+



In [18]:
#Agrupamiento por habitaciones, baños y zipcode
agrupado = df.groupBy("zipcode", "bedrooms", "bathrooms") \
    .agg(avg("price").alias("precio_promedio")) \
    .orderBy("zipcode", "bedrooms", "bathrooms")


In [19]:
#Visualización de agrupación
print("Agrupación por zipcode, habitaciones y baños:")
agrupado.show()

Agrupación por zipcode, habitaciones y baños:
+-------+--------+---------+------------------+
|zipcode|bedrooms|bathrooms|   precio_promedio|
+-------+--------+---------+------------------+
|  98001|       0|      0.0|          139950.0|
|  98001|       1|      1.0|          166000.0|
|  98001|       1|      2.0|          171000.0|
|  98001|       2|      1.0|197428.57142857142|
|  98001|       2|      1.5|          350000.0|
|  98001|       2|     1.75|          246112.5|
|  98001|       2|      2.5|          214100.0|
|  98001|       2|     2.75|          239475.0|
|  98001|       3|     0.75|          363000.0|
|  98001|       3|      1.0|205182.80952380953|
|  98001|       3|      1.5|          224108.5|
|  98001|       3|     1.75| 260531.0810810811|
|  98001|       3|      2.0|256841.42857142858|
|  98001|       3|     2.25|          265999.0|
|  98001|       3|      2.5| 308581.8604651163|
|  98001|       3|     2.75|          255000.0|
|  98001|       3|      3.0|          3095

In [20]:
#Cierre de spark
spark.stop()